In [ ]:
import  sys
import numpy as np
from datasets import load_dataset, DatasetDict, load_metric
from transformers import ViTForImageClassification, ViTImageProcessor, Trainer, TrainingArguments, EarlyStoppingCallback

sys.path.append('../')
from project.vit_training_utils import transform_data_for_ViT, collate_fn_for_ViT



## Notebook Purpose
<b> This is notebook number 3c </b>

To give code guidance on training ResNet models used in MixtureModel

### Notebook Order
1. getData
2. downloadData
3. trainResNetModel | trainPromptTransformerClassifier | trainViTClassifier
4. localMixtureEval

In [ ]:
dataset_path = '../data/may_eval_dataset/'

dataset = load_dataset('imagefolder', data_dir=dataset_path, split = "train", drop_labels = False)

In [ ]:
# Split the dataset into train and test sets
splits = dataset.train_test_split(test_size=0.12)
dataset = DatasetDict({
    'train': splits['train'],
    'test': splits['test']
})

In [ ]:
dataset['train'][10]['image']

In [ ]:
labels = dataset['train'].features['label'].names

load the pretrained model for finetuning

In [ ]:
# Load pre-trained model

base_model = 'google/vit-base-patch16-224'
model = ViTForImageClassification.from_pretrained(base_model,
                                                  num_labels=5,
                                                  id2label={str(i): c for i, c in enumerate(labels)},
                                                  label2id={c: str(i) for i, c in enumerate(labels)},
                                                 ignore_mismatched_sizes = True)


## Get processor
processor = ViTImageProcessor.from_pretrained(base_model)


## prepare dataset

In [ ]:
prepared_ds = dataset.with_transform(lambda x: transform_data_for_ViT(x, processor))

## prepare trainer

In [ ]:
metric = load_metric("accuracy")
def compute_metrics(p):
    return metric.compute(predictions=np.argmax(p.predictions, axis=1), references=p.label_ids)

In [ ]:

training_args = TrainingArguments(
  output_dir="../models/may_24/vit-finetuned",
  per_device_train_batch_size=16,
  evaluation_strategy="steps",
  num_train_epochs=2,
  fp16=True,
  save_steps=100,
  eval_steps=100,
  logging_steps=10,
  learning_rate=2e-5,
  save_total_limit=2,
  remove_unused_columns=False,
  push_to_hub=False,
  load_best_model_at_end=True,
 metric_for_best_model='accuracy',  # Use accuracy for selecting the best model
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=collate_fn_for_ViT,
    compute_metrics=compute_metrics,
    train_dataset=prepared_ds["train"],
    eval_dataset=prepared_ds["test"],
    tokenizer=processor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],  # Early stopping after 3 epochs of no improvement

)

## Train and save model

In [ ]:
train_results = trainer.train()
trainer.save_model()
trainer.log_metrics("train", train_results.metrics)
trainer.save_metrics("train", train_results.metrics)
trainer.save_state()